[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/area_transport/area_transport.ipynb)

# Surface areas through a collapse

Flatten a cube while sliding its layers sideways, and its volume goes to zero, yet its faces keep some area. This notebook follows the cube's edges, faces and volume through that deformation with maps that all grow from one. In numga such a map is an *extensor*: a geometric expression with one of its arguments left open, which then takes that argument later, composes with other maps, and batches over many cases at once like any other value. The deformation moves points; its *outermorphism* moves oriented patches; and its *adjugate* and *adjoint* carry measurements back, even through the flat sheet, where no inverse exists.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from itertools import product

import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import VGA3D
from examples.mechanics.area_transport import render
from examples.mechanics.area_transport.core import Surface

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()

# One entry per deformation, including the flat endpoint.
names = ("Solid", "Compressed and sheared", "Flat sheet")
side = 1.0
thicknesses = np.array([1.0, 0.35, 0.0])
shears = np.array([0.0, 0.45, 0.70])
# The cube's corners, and its faces' outward directions.
corners = mv.vector(list(product((-side / 2, side / 2), repeat=3)))             # [vertices] Vector
outward = concatenate([mv.basis(), -mv.basis()])                                # [faces] Vector

## 1. A deformation, and what it does to areas

Writing `Vector` where a vector would go leaves that argument open. `Vector + (t - 1) * mv.z * (mv.z | Vector)` is no vector but a map that takes one, `Vector <- Vector`: the identity, plus a squeeze along z; the deformation below adds a slide of each layer along x. A map on vectors also moves the plane two vectors span, by moving both edges, `areas(a ^ b) == deformation(a) ^ deformation(b)`: that map on planes, `Bivector <- Bivector`, is the deformation's outermorphism, and the factor it scales volumes by is its determinant. A face is an oriented patch, a bivector, or equally its area normal, the vector at right angles to it as long as its area.

In [ ]:
# --- math -----------------------------------------------------------------------------
# Each face's area normal, its outward direction as long as the face's area, and its oriented patch,
# the plane dual to it.
rest_normals = outward * side**2                                                # [faces] Vector
rest_patches = rest_normals.dual_inverse()                                      # [faces] Bivector

# Each layer keeps its x and y directions; its height controls the squeeze and sideways slide.
# The open Vector is the input slot: calling deformation(position) fills it.
deformation = (Vector + (thicknesses - 1) * mv.z * (mv.z | Vector)
               + shears * mv.x * (mv.z | Vector))                               # [cases] Vector <- Vector

# The images of both edges determine the image of their patch.
# areas(first ^ second) == deformation(first) ^ deformation(second)
areas = deformation.outermorphism(Bivector)                                     # [cases] Bivector <- Bivector
volume = deformation.det()                                                      # [cases] Scalar

## 2. Carrying measurements back: the adjugate and the adjoint

A direction and a patch together measure a volume, `direction & patch`, their regressive product. The adjugate of the area map carries a direction back so that it measures against an undeformed patch what it measures against the deformed one: `adjugate(c) & patch == c & areas(patch)`. It is defined by that measurement alone, with no inverse, so it exists at zero volume too, and the deformation after it multiplies by the volume, `deformation(adjugate(c)) == volume * c`. The adjoint does the same through the metric, with `|` for `&`: `adjugate.adjoint()` is the map that carries area normals forward with the faces.

In matrix notation the two maps read as $\operatorname{adj}(F)$ and $\operatorname{cof}(F)=\operatorname{adj}(F)^\mathsf{T}$. For invertible $F$, the latter is $\det(F)F^{-\mathsf{T}}$, as in [Nanson's relation](https://my.mech.utah.edu/~brannon/public/Deformation.pdf).

In [ ]:
# The area map's complementary pullback acts on vectors; it exists at zero volume too.
adjugate = areas.adjugate()                                                     # [cases] Vector <- Vector
cofactor = adjugate.adjoint()                                                   # [cases] Vector <- Vector

# Carry every face in both descriptions: an oriented patch and an area-weighted normal.
vertices = deformation[:, None](corners)                                        # [cases, vertices] Vector
centres = deformation[:, None](outward * (side / 2))                            # [cases, faces] Vector
patches = areas[:, None](rest_patches)                                          # [cases, faces] Bivector
area_normals = cofactor[:, None](rest_normals)                                  # [cases, faces] Vector
surface = Surface(vertices, centres, patches, area_normals)

## 3. The flat sheet

At zero volume the deformation loses a direction, and the adjugate points along it: `adjugate(mv.z)` is the direction the cube was flattened along, and the deformation sends it to nothing. The faces lying in the sheet keep their area, and the sheared side faces still cover part of it.

In [ ]:
# Follow the image of one volume measurement through every deformation.
# Applying the deformation afterward multiplies it by the volume ratio.
back = adjugate(mv.z)                                                           # [cases] Vector
forward = deformation(back)                                                     # [cases] Vector
# forward == volume * mv.z; at zero volume, back lies in the deformation's kernel.
render.comparison(surface, volume, names);

In [ ]:
# checks
# The adjugate keeps the volume a direction measures against a patch, the flat sheet included.
c, patch = mv.vector([0.3, -1.0, 0.5]), mv.bivector([1.0, 0.2, -0.4])
np.testing.assert_allclose((adjugate(c) & patch).kernel, (c & areas(patch)).kernel, atol=1e-12)
# The deformation after the adjugate multiplies by the volume: at the flat sheet, it sends the adjugate's image to nothing.
np.testing.assert_allclose((forward - volume * mv.z).kernel, 0.0, atol=1e-12)
# The adjoint keeps the metric pairing, and the carried area normals are the carried patches' duals.
np.testing.assert_allclose((cofactor(c) | patch.dual()).kernel, (c | adjugate(patch.dual())).kernel, atol=1e-12)
np.testing.assert_allclose((area_normals - patches.dual()).kernel, 0.0, atol=1e-12)


Run `python -m examples.mechanics.area_transport.scenarios` to save the figure and collapse animation in `plots/`.